# Crop and fertilizer research on Kaggle

**Setup:** In Kaggle, import this notebook. Create and attach a Kaggle dataset containing these four unchanged files from the repo’s `ML Research/ML-Week3/` folder: `Crop and fertilizer dataset.csv`, `Crop_recommendation.csv`, `run_experiments.py`, and `baseline_code.py`. Upload them together in one folder. The original scripts run on CPU; no GPU or internet is required if the dependencies are preinstalled.

Start with the 3-epoch smoke run on Dataset 2. For the complete Week 3 run, set `SMOKE_TEST = False` and rerun from Cell 1. The baseline is optional and runs separately.

## 1. Locate inputs and configure the run
Change `INPUT_DIR` from `None` to the specific Kaggle dataset directory if more than one attached folder has all four files.

In [ ]:
from pathlib import Path

INPUT_DIR = None  # Example: Path("/kaggle/input/datasets/your-name/your-dataset")
SMOKE_TEST = True
RUN_PAPER_BASELINES = False

files = ["Crop and fertilizer dataset.csv", "Crop_recommendation.csv",
         "run_experiments.py", "baseline_code.py"]
if INPUT_DIR is None:
    candidates = [p.parent for p in Path("/kaggle/input").rglob("run_experiments.py")
                  if all((p.parent / name).is_file() for name in files)]
    if len(candidates) != 1:
        raise ValueError(f"Expected one input folder with all four files; found {candidates}. Attach the files or set INPUT_DIR.")
    INPUT_DIR = candidates[0]
else:
    INPUT_DIR = Path(INPUT_DIR).resolve()
missing = [name for name in files if not (INPUT_DIR / name).is_file()]
if missing:
    raise FileNotFoundError(f"Missing in {INPUT_DIR}: {missing}")

from tempfile import mkdtemp
RUN_DIR = Path(mkdtemp(prefix="crop-research-", dir="/kaggle/working"))
print("Inputs:", INPUT_DIR)
print("Outputs:", RUN_DIR)

## 2. Check dependencies and inspect the data
Kaggle usually provides these packages. If imports fail, install the missing package in a separate cell with Internet enabled, or attach it as a wheel for offline installation. The Week 3 script uses XGBoost only if installed.

In [ ]:
import importlib.util
from importlib.metadata import version
from IPython.display import display
import pandas as pd
import sklearn, torch, matplotlib, openpyxl
# The original scripts detect categorical columns via object dtype.
# Preserve that behavior on pandas versions that infer a string dtype by default.
try:
    pd.set_option("future.infer_string", False)
except pd.errors.OptionError:
    pass

for package in ["pandas", "scikit-learn", "torch", "matplotlib", "openpyxl"]:
    print(package, version(package))
print("XGBoost installed:", importlib.util.find_spec("xgboost") is not None)

DATASET_1, DATASET_2 = (INPUT_DIR / name for name in files[:2])
TASKS = [(DATASET_1, "Dataset 1 (Crop & Fertilizer)", "Fertilizer"),
         (DATASET_1, "Dataset 1 (Crop & Fertilizer)", "Crop"),
         (DATASET_2, "Dataset 2 (Crop Recommendation)", "label")]
for path, name, target in TASKS:
    df = pd.read_csv(path)
    counts = df[target].value_counts()
    print(f"{name} / {target}: {len(df)} rows, {len(counts)} classes, "
          f"smallest class = {counts.min()}, exact duplicate rows = {df.duplicated().sum()}")

## 3. Run the Week 3 contributions
C1: agronomic feature engineering. C2: crop-conditioned fertilizer prediction (fertilizer only). C3: gated residual network with class-balanced learning. C4: uncertainty-aware fusion. Each task uses a stratified train/validation/test split within its own dataset.

In [ ]:
import os
import shutil
from functools import partial

# Copy scripts to a writable run directory; never modify attached inputs.
for name in files[2:]:
    shutil.copy2(INPUT_DIR / name, RUN_DIR / name)
os.chdir(RUN_DIR)

def load_script(name):
    spec = importlib.util.spec_from_file_location(name, RUN_DIR / f"{name}.py")
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

week3 = load_script("run_experiments")
week3.OUTPUT_DIR = RUN_DIR / "week3_results"
week3.OUTPUT_DIR.mkdir(exist_ok=True)
if SMOKE_TEST:
    # The function's original epoch default was bound at import time.
    week3.train_network = partial(week3.train_network, epochs=3)
week3.set_seed(week3.SEED)
week3.save_architecture_diagrams(week3.OUTPUT_DIR)

records = []
for path, name, target in (TASKS[-1:] if SMOKE_TEST else TASKS):
    for result in week3.run_experiment(path, name, target):
        records.append({"Dataset": name, "Target": target, **result})
summary = pd.DataFrame(records)
summary.to_csv(week3.OUTPUT_DIR / "kaggle_summary.csv", index=False)
display(summary[["Dataset", "Target", "Contribution", "Model", "Accuracy", "Macro_F1"]]
        .sort_values(["Dataset", "Target", "Macro_F1"], ascending=[True, True, False]))

## 4. Optional: run the original three-model baseline
Set `RUN_PAPER_BASELINES = True` in Cell 1 before running this cell. This runs the 1D-CNN, dual-branch fusion network, and gradient boosting on all three tasks. In smoke mode the neural models run for 3 epochs. This is a separate protocol, not a like-for-like comparison with Week 3.

In [ ]:
if RUN_PAPER_BASELINES:
    for path in (DATASET_1, DATASET_2):
        shutil.copy2(path, RUN_DIR / path.name)
    baseline = load_script("baseline_code")
    baseline.EPOCHS = 3 if SMOKE_TEST else 150
    baseline.main()
else:
    print("Baseline skipped. Set RUN_PAPER_BASELINES = True to enable it.")

## 5. Download generated results
The archive contains the results from this run, including the Week 3 summary and plots.

In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED
from IPython.display import FileLink

archive = RUN_DIR.with_suffix(".zip")
with ZipFile(archive, "w", compression=ZIP_DEFLATED) as z:
    for directory in ("week3_results", "analysis_outputs", "outputs"):
        for path in sorted((RUN_DIR / directory).rglob("*")):
            if path.is_file():
                z.write(path, path.relative_to(RUN_DIR))
print(archive)
display(FileLink(os.path.relpath(archive, start=Path.cwd())))

## What to watch when interpreting results
- Dataset 1 has 4,513 rows, 16 crop classes, and 19 fertilizer classes. The rarest fertilizer class has just six rows. Report macro F1 and per-class support, not accuracy alone.
- The unmodified Week 3 feature selector retains `Fertilizer` when predicting `Crop`, and `Crop` when predicting `Fertilizer`. This is appropriate only if that field is genuinely known before inference. For a sensor-only evaluation, exclude the counterpart label from the input features; retain the crop labels separately during training for C2. Do not compare the two protocols as if their inputs were identical.
- `SENSOR_NOISE` is declared in Week 3 but not used in `run_experiment`. The baseline does inject noise before splitting. Do not interpret the Week 3 run as a 25% noise-robustness test or directly compare its scores with the baseline.
- A random row split does not establish generalization to new districts or farms. This notebook runs the available Week 3 implementation; it does not regenerate archived ablations or noise sweeps whose runners are not provided by this entry point.